# Natural Language Processing (NLP) Template


### Step 1: Load Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (6, 4)

# Load dataset here

# Turn output variable into 1/0 value


df


### Step 2: Data Cleanup


In [ ]:
import re

def basic_clean(text: str) -> str:
    # Lowercase
    text = text.lower()
    # Remove anything that's not a letter or space
    text = re.sub(r"[^a-z\s]", "", text)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Cleanup the text data

df

### Step 3: Create Bag-of-Words


In [ ]:
# Vectorize using Bag-of-Words
vectorizer = CountVectorizer(stop_words="english")
X_bow = vectorizer.fit_transform(df["clean_comment"])

X_bow.shape  # (n_documents, n_features)
print(f"Number of documents: {X_bow.shape[0]}")
print(f"Number of unique words (features): {X_bow.shape[1]}")
feature_names = vectorizer.get_feature_names_out()
print("Sample feature names:", feature_names[:10])

print(X_bow.toarray()) 

In [ ]:
# See some feature names (or column names)
feature_names = vectorizer.get_feature_names_out()
feature_names[:20]


### Step 4: Create a simple sentiment classifier

Now that we have numeric features from text, we can build a simple classifier to predict sentiment (positive/negative) based on course comments.


In [ ]:
# Our dependent variable.
y = ?

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_bow, y, test_size=0.3, random_state=42
)

# Train a simple Logistic Regression classifier that uses the Bag-of-Words features (columns)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# How well does it do on the training set?
y_train_pred = clf.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
print(f"Training accuracy: {train_acc}")

# How well does it do on the *test* set?
y_pred = clf.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_acc}")

# Confusion Matrix for test set
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
disp.plot()
plt.title("Sentiment Classification – Confusion Matrix")
plt.show()



### Step 5: TF-IDF 


In [ ]:
# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
X_tfidf = tfidf_vectorizer.fit_transform(df["clean_comment"])

X_tfidf.shape
X_tfidf.toarray()

In [ ]:
# Compare the Bag-of-Words and TF-IDF features against each other, showing 
# the first 5 rows and first 10 columns of each
print("Bag-of-Words feature matrix (first 5 rows, 10 columns):")
print(X_bow.toarray()[:5, :10])
print("\nTF-IDF feature matrix (first 5 rows, 10 columns):")
print(X_tfidf.toarray()[:5, :10])

In [ ]:
# Do the same train/test split and classification with TF-IDF features
# Use logistical regression again and compare accuracy.
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_tfidf, y, test_size=0.3, random_state=42
)

clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train_t, y_train_t)

# How well does it do on the training set?
y_train_pred = clf_tfidf.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
print(f"Training accuracy: {train_acc}")

# How well does it do on the *test* set?
y_pred = clf_tfidf.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_acc}")

# Confusion Matrix for test set
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf_tfidf.classes_)
disp.plot()
plt.title("Sentiment Classification – Confusion Matrix")
plt.show()


### Step 6: Clustering to explore topics

In [ ]:
k = 2  # we suspect there might be "positive" vs "negative"-like groups
kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
clusters = kmeans.fit_predict(X_tfidf)

df["cluster"] = clusters
df[["comment", "cluster"]]


In [ ]:
terms = tfidf_vectorizer.get_feature_names_out()
centers = kmeans.cluster_centers_

for cluster_id in range(k):
    print(f"\nCluster {cluster_id}")
    center = centers[cluster_id]
    top_idx = center.argsort()[::-1][:10]
    print([terms[i] for i in top_idx])


In [ ]:
# Visualize clusters using PCA (2D)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tfidf.toarray())
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis')
plt.title("K-Means Clustering of Course Comments (PCA-reduced)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.show()